# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library, referencing entities only by their `@id` attributes as per the Croissant schema.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and discover the available record sets, using the Croissant schema URL.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object (not by subscripting)
print("Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Identifier (DOI):", dataset.metadata.identifier)
print("Version:", getattr(dataset.metadata, 'version', None))
print("License:", dataset.metadata.license)


## 2. Data Overview
Review available record sets, their fields, and key attributes, referencing all by their `@id` fields.


In [ ]:
# List all record sets and their fields by @id
print("Record Sets found in this Croissant schema:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}  (name: {getattr(rs, 'name', 'n/a')})")
    print("  Fields in this RecordSet:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} (name: {getattr(field, 'name', 'n/a')}, dataType: {getattr(field, 'data_type', 'n/a')})")
    print()

## 3. Data Extraction
We now load data from each record set, referencing only their `@id`s. All dataframes are collected in a dictionary by record set `@id`.


In [ ]:
# Prepare to extract records for each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"All record set @id's: {record_set_ids}")

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    # Some record sets might be empty or not parseable
    if records:
        df = pd.DataFrame(records)
        print(f"RecordSet @id: {rs_id}, columns: {list(df.columns)}")
        dataframes[rs_id] = df
    else:
        print(f"RecordSet @id: {rs_id} yielded no records.")

# For demonstration, automatically choose the first non-empty record set for analysis
active_record_set = None
for rs_id, df in dataframes.items():
    if not df.empty:
        active_record_set = rs_id
        break

if active_record_set is not None:
    print(f"\nProceeding with active_record_set @id: {active_record_set}")
    print("Columns in this DataFrame:", dataframes[active_record_set].columns.tolist())
    display(dataframes[active_record_set].head())
else:
    print("No non-empty record sets with records found!")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field in the active record set (referenced by its `@id`), filter for significant values, normalize, and group by a categorical field.


In [ ]:
if active_record_set is not None:
    df = dataframes[active_record_set]
    # Find numeric fields by their type from schema
    numeric_fields = []
    group_fields = []
    for rs in dataset.metadata.record_sets:
        if rs.id == active_record_set:
            for field in rs.fields:
                if ('Float' in str(getattr(field, 'data_type', ''))) or (getattr(field, 'data_type', '') in ['Float', 'Number', 'Integer']):
                    numeric_fields.append(field.id)
                if getattr(field, 'data_type', '') == 'Text':
                    group_fields.append(field.id)
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Found numeric field @id: {numeric_field}")
    else:
        print("No numeric field found in this record set.")

    # Ensure column exists
    if numeric_fields and numeric_field in df.columns:
        threshold = df[numeric_field].dropna().quantile(0.5) # use median as a sample filter threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[numeric_field + '_normalized'] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Try grouping by a text field
        group_field = None
        for gf in group_fields:
            if gf in df.columns and df[gf].nunique() < 10:
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by field @id: {group_field} (mean of {numeric_field}):")
            display(grouped_df)
        else:
            print("No suitable text field found for grouping (with < 10 unique values). Showing top values by numeric field:")
            display(filtered_df[[numeric_field]].sort_values(numeric_field, ascending=False).head())
    else:
        print("Active record set does not have numeric field data for EDA.")
else:
    print("No active record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. Example: histogram and boxplot of the selected numeric field, using valid `@id` field references.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if active_record_set is not None and numeric_fields and numeric_field in df.columns:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=ax[0], color='tab:blue')
    ax[0].set_title(f"Distribution of {numeric_field} (@id)")

    sns.boxplot(x=df[numeric_field].dropna(), ax=ax[1], color='tab:orange')
    ax[1].set_title(f"Boxplot of {numeric_field} (@id)")

    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we:
- Loaded the Croissant metadata and listed available record sets and their fields by `@id`.
- Loaded data from all record sets, exploring their structure using only Croissant references.
- Performed filtering, normalization, and basic grouping on a numeric field, referencing by canonical `@id` as required by best practice.
- Visualized the main quantitative attribute in the record set.

This approach demonstrates a fully FAIR (Findable, Accessible, Interoperable, Reusable) workflow for data science with Croissant datasets and the `mlcroissant` library.